# Age Classification — Maximum Performance Training

**Task:** Binary classification (Young vs Old) using ResNet-18.

**Techniques implemented:**
- Heavy head architecture (Linear → BN → GELU → Dropout → Linear)
- Built-in Test Time Augmentation (horizontal flip at inference)
- Strong data augmentation (RandomResizedCrop, ColorJitter, RandAugment, RandomErasing)
- MixUp & CutMix regularisation
- Label smoothing Cross-Entropy + optional teacher embedding guidance
- Adam optimizer with OneCycleLR scheduler
- Exponential Moving Average (EMA) of weights
- Mixed-precision training (AMP) + gradient clipping
- BatchNorm calibration after EMA
- Phase 2: retrain on train+valid combined data


In [1]:
# Install required packages if not already installed
import subprocess
import sys

def install_package(pip_name, import_name=None):
    """Install package using pip_name, check import using import_name (or pip_name if None)."""
    if import_name is None:
        import_name = pip_name
    
    try:
        __import__(import_name)
        print(f"✓ {pip_name} is already installed")
    except ImportError:
        print(f"Installing {pip_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])
        print(f"✓ {pip_name} installed successfully!")

# Install required packages (pip_name, import_name)
required_packages = [
    ('numpy', 'numpy'),
    ('torch', 'torch'),
    ('torchvision', 'torchvision'),
    ('pillow', 'PIL'),
]

for pip_name, import_name in required_packages:
    install_package(pip_name, import_name)

print("\nAll required packages are ready!")

Installing numpy...
✓ numpy installed successfully!
Installing torch...
✓ torch installed successfully!
Installing torchvision...
✓ torchvision installed successfully!
✓ pillow is already installed

All required packages are ready!


In [ ]:
import os, random, platform, shutil, csv
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torch.amp import GradScaler, autocast

from model_class import build_model
from training_utils import (
    get_advanced_augmentation, get_eval_transform,
    mixup_data, cutmix_data, mixup_criterion,
    LabelSmoothingCrossEntropy, CombinedLoss,
    EMA, calibrate_bn,
)

# -- Reproducibility --
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True       # faster convolutions
torch.backends.cudnn.deterministic = False   # allow non-deterministic ops for speed

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = torch.cuda.is_available()
print(f'Device: {DEVICE}  |  AMP: {USE_AMP}')


Device: cpu  |  AMP: False


In [ ]:
# ==================== Configuration ====================

DATA_ROOT        = 'dataset/'
TRAIN_DIR        = os.path.join(DATA_ROOT, 'train')
VALID_DIR        = os.path.join(DATA_ROOT, 'valid')
VALID_LABELS     = os.path.join(DATA_ROOT, 'valid_labels.csv')
TEACHER_EMB_PATH = os.path.join('teacher_embeddings', 'train_embeddings.pt')
IMG_SIZE         = 224
IMG_EXT          = ('.png', '.jpg', '.jpeg')

# -- Hyperparameters --
BATCH_SIZE    = 64
NUM_EPOCHS    = 100
LEARNING_RATE = 3e-3        # max LR for OneCycleLR
WEIGHT_DECAY  = 0.05        # Adam weight decay
GRAD_CLIP     = 1.0         # max grad norm

# -- Augmentation --
MIXUP_ALPHA   = 0.4
CUTMIX_ALPHA  = 1.0
MIX_PROB      = 0.5         # probability of applying MixUp/CutMix per batch

# -- Loss --
LABEL_SMOOTH  = 0.1
LOSS_ALPHA    = 0.6         # weight on CE when teacher guidance is active

# -- EMA --
EMA_DECAY     = 0.999

# -- Workers (0 on Windows to avoid multiprocessing issues) --
NUM_WORKERS = 0 if platform.system() == 'Windows' else 4

# -- Teacher embeddings --
USE_TEACHER_GUIDANCE = False
teacher_embeddings   = None
if os.path.isfile(TEACHER_EMB_PATH):
    teacher_embeddings = torch.load(TEACHER_EMB_PATH, map_location='cpu')
    USE_TEACHER_GUIDANCE = True
    print(f'Loaded {len(teacher_embeddings)} teacher embeddings')
else:
    print('No teacher embeddings found -- training without guidance')

# -- Submission --
ROLL_NO = 'b23es1001'   

print(f'\nEpochs: {NUM_EPOCHS}  |  LR: {LEARNING_RATE}  |  WD: {WEIGHT_DECAY}')
print(f'MixUp a: {MIXUP_ALPHA}  |  CutMix a: {CUTMIX_ALPHA}  |  Mix prob: {MIX_PROB}')
print(f'EMA decay: {EMA_DECAY}  |  Grad clip: {GRAD_CLIP}  |  AMP: {USE_AMP}')
print(f'Workers: {NUM_WORKERS}')


No teacher embeddings found -- training without guidance

Epochs: 100  |  LR: 0.003  |  WD: 0.05
MixUp a: 0.4  |  CutMix a: 1.0  |  Mix prob: 0.5
EMA decay: 0.999  |  Grad clip: 1.0  |  AMP: False
Workers: 0


In [4]:
# -- Transforms --
train_transform = get_advanced_augmentation(IMG_SIZE)
valid_transform = get_eval_transform(IMG_SIZE)

print('Train:', train_transform)
print('Valid:', valid_transform)


Train: Compose(
    RandomResizedCrop(size=(224, 224), scale=(0.8, 1.0), ratio=(0.9, 1.1), interpolation=bilinear, antialias=True)
    RandomHorizontalFlip(p=0.5)
    RandomApply(
    p=0.3
    ColorJitter(brightness=(0.8, 1.2), contrast=(0.8, 1.2), saturation=(0.85, 1.15), hue=(-0.05, 0.05))
)
    RandomGrayscale(p=0.05)
    RandAugment(interpolation=InterpolationMode.NEAREST, num_ops=2, magnitude=9, num_magnitude_bins=31)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    RandomErasing(p=0.15, scale=(0.02, 0.15), ratio=(0.3, 3.3), value=0, inplace=False)
)
Valid: Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
    Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
)


In [ ]:
# ==================== Datasets & Loaders ====================

class TrainDataset(Dataset):
    def __init__(self, root, transform=None):
        self.transform = transform
        self.samples = []
        for label in [0, 1]:
            cls_dir = os.path.join(root, str(label))
            for fname in sorted(os.listdir(cls_dir)):
                if fname.lower().endswith(IMG_EXT):
                    self.samples.append((os.path.join(cls_dir, fname), label, fname))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label, fname = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label, fname


class ValidDataset(Dataset):
    def __init__(self, root, labels_csv, transform=None):
        self.transform = transform
        self.samples = []
        labels_dict = {}
        with open(labels_csv, 'r') as f:
            for row in csv.DictReader(f):
                labels_dict[row['image']] = int(row['label'])
        for fname in sorted(os.listdir(root)):
            if fname.lower().endswith(IMG_EXT) and fname in labels_dict:
                self.samples.append((os.path.join(root, fname), labels_dict[fname], fname))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label, fname = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform:
            img = self.transform(img)
        return img, label, fname


train_dataset = TrainDataset(TRAIN_DIR, transform=train_transform)
valid_dataset = ValidDataset(VALID_DIR, VALID_LABELS, transform=valid_transform)

_loader_kw = dict(pin_memory=torch.cuda.is_available(), num_workers=NUM_WORKERS)
if NUM_WORKERS > 0:
    _loader_kw['persistent_workers'] = True

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          drop_last=True, **_loader_kw)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False,
                          **_loader_kw)

print(f'Train: {len(train_dataset)} images  ({len(train_loader)} batches)')
print(f'Valid: {len(valid_dataset)} images  ({len(valid_loader)} batches)')


Train: 18332 images  (286 batches)
Valid: 134 images  (3 batches)


In [ ]:
# ==================== Model / Optimizer / Loss / EMA / AMP ====================

model = build_model(num_classes=2).to(DEVICE)

# -- Loss --
ce_criterion = LabelSmoothingCrossEntropy(smoothing=LABEL_SMOOTH).to(DEVICE)
combined_criterion = None
if USE_TEACHER_GUIDANCE:
    combined_criterion = CombinedLoss(
        alpha=LOSS_ALPHA,
        label_smoothing=LABEL_SMOOTH,
        teacher_embeddings=teacher_embeddings,
        student_feat_dim=256,
    ).to(DEVICE)

# -- Optimizer (include projection params if they exist) --
params = list(model.parameters())
if (combined_criterion is not None
        and hasattr(combined_criterion, 'projection')
        and combined_criterion.projection is not None):
    params += list(combined_criterion.projection.parameters())

optimizer = optim.Adam(params, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# -- Scheduler --
steps_per_epoch = len(train_loader)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=LEARNING_RATE,
    total_steps=NUM_EPOCHS * steps_per_epoch,
    pct_start=0.1,
    anneal_strategy='cos',
)

# -- EMA + AMP --
ema    = EMA(model, decay=EMA_DECAY)
scaler = GradScaler('cuda', enabled=USE_AMP) if USE_AMP else None

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parameters: {n_params:,}')
print(f'Optimizer : Adam (lr={LEARNING_RATE}, wd={WEIGHT_DECAY})')
print(f'Scheduler : OneCycleLR ({NUM_EPOCHS} epochs x {steps_per_epoch} steps)')
print(f'Loss      : {"Combined (a=" + str(LOSS_ALPHA) + ")" if USE_TEACHER_GUIDANCE else "Label-smoothed CE"}')
print(f'EMA decay : {EMA_DECAY}')


Parameters: 11,308,866
Optimizer : AdamW (lr=0.003, wd=0.05)
Scheduler : OneCycleLR (100 epochs x 286 steps)
Loss      : Label-smoothed CE
EMA decay : 0.999


C:\Users\Aashcharya\AppData\Local\Temp\ipykernel_66028\1557966881.py:37: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=USE_AMP)


In [8]:
# ==================== Training Helpers ====================

def train_one_epoch(model, loader, optimizer, scheduler, ce_crit,
                    combined_crit, ema_obj, scaler, device,
                    use_teacher=False):
    """One epoch: MixUp/CutMix + AMP + grad clip + EMA."""
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for images, labels, fnames in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        # -- Choose augmentation strategy --
        is_mixed = False
        if random.random() < MIX_PROB:
            is_mixed = True
            if random.random() < 0.5:
                images, la, lb, lam, _ = mixup_data(images, labels, MIXUP_ALPHA)
            else:
                images, la, lb, lam, _ = cutmix_data(images, labels, CUTMIX_ALPHA)

        # -- Forward + loss --
        if USE_AMP:
            with autocast('cuda', enabled=True):
                if is_mixed:
                    out = model(images)
                    loss = lam * ce_crit(out, la) + (1 - lam) * ce_crit(out, lb)
                elif use_teacher and combined_crit is not None:
                    out, feat = model(images, return_features=True)
                    loss, _, _ = combined_crit(out, labels, feat, fnames)
                else:
                    out = model(images)
                    loss = ce_crit(out, labels)
        else:
            if is_mixed:
                out = model(images)
                loss = lam * ce_crit(out, la) + (1 - lam) * ce_crit(out, lb)
            elif use_teacher and combined_crit is not None:
                out, feat = model(images, return_features=True)
                loss, _, _ = combined_crit(out, labels, feat, fnames)
            else:
                out = model(images)
                loss = ce_crit(out, labels)

        # -- Backward + step --
        optimizer.zero_grad(set_to_none=True)
        if USE_AMP and scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            optimizer.step()
        scheduler.step()
        ema_obj.update(model)

        # -- Metrics --
        total_loss += loss.item() * images.size(0)
        correct += (out.argmax(1) == labels).sum().item()
        total += images.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def validate(model, loader, device):
    """Evaluate accuracy (TTA is built into model.eval forward)."""
    model.eval()
    correct, total = 0, 0
    for images, labels, _ in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        out = model(images)
        correct += (out.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return correct / total


# ==================== Phase 1: Train + Validate ====================

best_val_acc = 0.0

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(
        model, train_loader, optimizer, scheduler,
        ce_criterion, combined_criterion, ema, scaler, DEVICE,
        use_teacher=USE_TEACHER_GUIDANCE,
    )

    # Validate with EMA weights (+ TTA inside model.eval)
    ema.apply_shadow(model)
    val_acc = validate(model, valid_loader, DEVICE)

    saved = ''
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model, 'best_model.pth')
        saved = f'  ** BEST {val_acc*100:.2f}%'

    ema.restore(model)

    lr = scheduler.get_last_lr()[0]
    if epoch % 5 == 0 or epoch <= 5 or saved:
        print(f'Epoch {epoch:03d}/{NUM_EPOCHS}  '
              f'Loss {train_loss:.4f}  Train {train_acc:.4f}  '
              f'Val {val_acc:.4f}  LR {lr:.6f}{saved}')

print(f'\nPhase 1 done -- best validation accuracy: {best_val_acc*100:.2f}%')

# ==================== Save Final Model for Submission ====================

# Load the best model that was saved during training
print('\nPreparing final model for submission...')
if os.path.exists('best_model.pth'):
    best_model = torch.load('best_model.pth', map_location=DEVICE)
    # Apply EMA weights (we need to recreate EMA state, but for simplicity,
    # we'll just use the saved model which should have good weights)
    # Note: The saved model is already the best validation model
    final_model = best_model
else:
    # Fallback: use current model with EMA applied
    final_model = model
    ema.apply_shadow(final_model)

# Calibrate BatchNorm with EMA weights
print('Calibrating BatchNorm with EMA weights...')
calibrate_bn(final_model, train_loader, DEVICE, steps=80)

# Fix module path for torch.load unpickling
final_model.__class__.__module__ = ROLL_NO

# Save final model
save_path = f'{ROLL_NO}.pth'
torch.save(final_model, save_path)
mb = os.path.getsize(save_path) / 1e6
print(f'\nSaved model to {save_path} ({mb:.1f} MB)')

# Copy model_class.py -> ROLL_NO.py
shutil.copy('model_class.py', f'{ROLL_NO}.py')
print(f'Copied model_class.py -> {ROLL_NO}.py')

print(f'\nPhase 1 submission files ready:')
print(f'  - {ROLL_NO}.pth')
print(f'  - {ROLL_NO}.py')
print(f'  - {ROLL_NO}.pdf (create this manually)')


KeyboardInterrupt: 